# k-clique density on Facebook

This notebook explores **k-clique count** and **k-clique density** on the Facebook ground-truth graph. The goal is to distinguish triangle closure from genuinely dense k-node structure. we'll use 4-clique as a starting point.

## Metric definition

Let $K_4(G)$ be the set of 4-cliques in $G$. Then

$$\delta_4(G)=\frac{|K_4(G)|}{\binom{|V|}{4}}.$$

Interpretation: among all 4-node subsets, how often do we see a fully connected group of four?

In [1]:

from pathlib import Path
import sys
import math
import itertools
import shutil

import networkx as nx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / 'metrics.py').exists():
    NOTEBOOK_DIR = NOTEBOOK_DIR.parent

sys.path.insert(0, str(NOTEBOOK_DIR))
from metrics import (
    load_graph,
    unique_triangle_count,
    wedge_count,
    gcc,
    alcc,
    count_k_cliques,
    k_clique_density,
    higher_order_global_clustering,
    higher_order_average_local_clustering,
    higher_order_local_clustering,
    node_k_clique_membership_counts,
    gcd11,
    orca_node_orbits_4,
    gcm11_from_orbits,
    GCD11_ORBITS,
)

DATA_PATH = NOTEBOOK_DIR.parent / 'data' / 'gt_txt' / 'facebook.txt'
G = load_graph(DATA_PATH)
print(f'Loaded {DATA_PATH.name}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')


Loading graph: 88234it [00:00, 691652.11it/s]


Loaded facebook.txt: 4039 nodes, 88234 edges


In [2]:

def induced_ego_subgraph(G, center=None, radius=1, max_nodes=40):
    if center is None:
        center = max(G.degree, key=lambda x: x[1])[0]
    nodes = set(nx.single_source_shortest_path_length(G, center, cutoff=radius).keys())
    H = G.subgraph(nodes).copy()
    if H.number_of_nodes() > max_nodes:
        nbrs = sorted(H.nodes(), key=lambda u: (-G.degree[u], u))[:max_nodes]
        H = G.subgraph(nbrs).copy()
    return center, H


def edge_df(G):
    return pd.DataFrame(sorted((min(u,v), max(u,v)) for u,v in G.edges()), columns=['u','v'])


In [ ]:
d4, k4 = k_clique_density(G, 4) # k_clique_density returns a tuple of (density, count)
pd.DataFrame([
    {'metric': '4-clique count', 'value': k4},
    {'metric': '4-clique density', 'value': d4},
])

Processing nodes: 100%|██████████| 4039/4039 [00:46<00:00, 87.62it/s]  


,metric,value
0,4-clique count,3.000467e+07
1,4-clique density,2.709880e-06


## Small subgraph example

On a small induced subgraph, we can list 4-cliques explicitly.

In [ ]:
center, H = induced_ego_subgraph(G, radius=1, max_nodes=20)
print('center node:', center)
print('subgraph nodes:', H.number_of_nodes(), 'edges:', H.number_of_edges())

In [ ]:
k4s = [tuple(c) for c in nx.enumerate_all_cliques(H) if len(c) == 4]
print('number of 4-cliques in small subgraph:', len(k4s))
pd.DataFrame({'4-clique': k4s[:20]})

In [ ]:
sub_k4 = count_k_cliques(H, 4)
sub_d4 = k_clique_density(H, 4)
pd.DataFrame([
    {'metric': 'subgraph 4-clique count', 'value': sub_k4},
    {'metric': 'subgraph 4-clique density', 'value': sub_d4},
])

In [ ]:
if k4s:
    clique_nodes = list(k4s[0])
    S = H.subgraph(clique_nodes).copy()
    pos = nx.spring_layout(S, seed=3)
    plt.figure(figsize=(4,4))
    nx.draw_networkx(S, pos=pos, node_size=500)
    plt.title('One 4-clique from the small subgraph')
    plt.axis('off')
    plt.show()
else:
    print('No 4-clique found in this small subgraph; try increasing max_nodes.')

## Interpretation

4-clique density is stricter than triangle-based clustering. A graph can have many triangles and still have relatively little fully connected 4-node structure.

For realism testing later, matching 4-clique density means reproducing the prevalence of genuinely dense 4-node groups, not just wedge closure.